Use the Python 3.11 kernel and private preprocessed data; see [setup](../../README.md). Run cells top to bottom.

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from ai_cpet_ots.dataloaders import (
    Model_1_2_dataset,
    Model_1_3_4_dataset,
    create_dataset_model_3,
    create_dataset_model_4,
    load_clinical_data,
    load_CPET_data,
)
from ai_cpet_ots.helpers import collate_model_1_3_4
from ai_cpet_ots.models import Model_1, Model_1_2, Model_1_3_4, Model_2, Model_3, Model_4
from ai_cpet_ots.paths import (
    BBB_CONTROL_DIR,
    BBB_OTS_DIR,
    CLIN_CONTROL_DIR,
    CLIN_OTS_DIR,
    CPET_CONTROL_DIR,
    CPET_OTS_DIR,
    ECG_CONTROL_DIR,
    ECG_OTS_DIR,
    MODEL_WEIGHTS,
    output_path,
    require_file,
)
from ai_cpet_ots.testers import test_model_1_2, test_model_1_3_4

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")

# Model-1-2 (Clinical + CPET)

In [ ]:
(
    _,
    _,
    X_train_clinical,
    Y_train_clinical,
    X_test_clinical,
    Y_test_clinical,
    trainset_idx,
    testset_idx,
) = load_clinical_data(CLIN_CONTROL_DIR, CLIN_OTS_DIR, return_indexes=True)

In [ ]:
_, _, X_train_cpet, Y_train_cpet, X_test_cpet, Y_test_cpet, trainset_idx, testset_idx = (
    load_CPET_data(CPET_CONTROL_DIR, CPET_OTS_DIR, return_indexes=True)
)

In [ ]:
X_train = np.concatenate((X_train_clinical, X_train_cpet), axis=1)
Y_train = Y_train_clinical
X_test = np.concatenate((X_test_clinical, X_test_cpet), axis=1)
Y_test = Y_test_clinical
print(X_train.shape, Y_train.shape)
print(X_test.shape, Y_test.shape)

In [ ]:
model = Model_1_2().to(device)
model_1 = Model_1().to(device)
model_2 = Model_2().to(device)

model.load_state_dict(
    torch.load(MODEL_WEIGHTS / "Model_1_2.pth", map_location=device, weights_only=True)
)
model_1.load_state_dict(
    torch.load(MODEL_WEIGHTS / "Model_1.pth", map_location=device, weights_only=True)
)
model_2.load_state_dict(
    torch.load(MODEL_WEIGHTS / "Model_2.pth", map_location=device, weights_only=True)
)

model.eval()
model_1.eval()
model_2.eval()

In [ ]:
"""Test model"""
testset = Model_1_2_dataset(X_test_clinical, Y_test_clinical, X_test_cpet, Y_test_cpet)
testloader = DataLoader(testset, batch_size=len(testset), shuffle=False)
auroc, auprc, f1, acc, y_preds, y_batch = test_model_1_2(
    model, model_1, model_2, "Model 1-2", testloader, False, True, True, True, True, True
)
print(
    f"AUROC: {auroc:.3f}, AUCPR: {auprc:.3f}, F1: {f1:.3f}, ACC: {acc:.3f}, Threshold {model.threshold:.3f}"
)

In [ ]:
"""Create df with testset predictions"""
c_clinical = pd.read_excel(CLIN_CONTROL_DIR)
o_clinical = pd.read_excel(CLIN_OTS_DIR)

c_clinical.sort_values(by="testID", ascending=True, inplace=True)
o_clinical.sort_values(by="testID", ascending=True, inplace=True)

clinical_data_id_validation = np.vstack((c_clinical, o_clinical))[testset_idx]

# Convert stacked ids to df
pred_df = (
    pd.DataFrame(clinical_data_id_validation).rename(columns={0: "ID"}).drop(columns=[1, 2, 3, 4])
)

# Create df for control ground truths
control_ids_df = pd.DataFrame(c_clinical)["testID"]
control_ids_df = pd.concat(
    [control_ids_df, pd.Series(np.zeros_like(c_clinical)[:, 0])], axis=1
).rename(columns={"testID": "ID", 0: "Ground truth"})

# Create df for OTS ground truths
ots_ids_df = pd.DataFrame(o_clinical)["testID"]

# Add ground truths to pred_df
pred_df = pred_df.merge(control_ids_df, on="ID", how="left")
pred_df.loc[pred_df["ID"].isin(ots_ids_df), "Ground truth"] = 1

# Append raw predictions
y_preds_numpy = y_preds.detach().cpu().numpy()
pred_df = pd.concat([pred_df, pd.Series(y_preds_numpy)], axis=1).rename(
    columns={0: "Model-1-2-raw"}
)

# Append model predictions
y_preds_bin = [1 if val >= model.threshold else 0 for val in y_preds_numpy]
pred_df = pd.concat([pred_df, pd.Series(y_preds_bin).rename("Model-1-2-pred")], axis=1)
pred_df

In [ ]:
np.testing.assert_array_equal(pred_df["Model-1-2-pred"], y_preds_bin)
np.testing.assert_array_equal(
    np.array(pred_df["Ground truth"]), np.array(y_batch.detach().cpu().numpy())
)
pd.testing.assert_series_equal(
    pd.DataFrame(clinical_data_id_validation)[0],
    pred_df["ID"],
    check_names=False,
)

# Model 1-3-4 (Clinical + BBB + ECG)

In [ ]:
(
    _,
    _,
    X_train_clinical,
    Y_train_clinical,
    X_test_clinical,
    Y_test_clinical,
    trainset_idx,
    testset_idx,
) = load_clinical_data(CLIN_CONTROL_DIR, CLIN_OTS_DIR, return_indexes=True)

In [ ]:
"""Load BBB data for model 3"""
_, _, X_train_bbb, Y_train_bbb, X_test_bbb, Y_test_bbb, _, variables = create_dataset_model_3(
    BBB_CONTROL_DIR, BBB_OTS_DIR, mode="minmax"
)

In [ ]:
"""Load ECG data for model 4"""
X_train_ecg, Y_train_ecg, X_test_ecg, Y_test_ecg = create_dataset_model_4(
    ECG_CONTROL_DIR, ECG_OTS_DIR, filter_mode="pan-tompkins"
)

In [ ]:
model_1 = Model_1().to(device)
model_1.load_state_dict(
    torch.load(MODEL_WEIGHTS / "Model_1.pth", map_location=device, weights_only=True)
)
model_1.eval()

model_bbb = Model_3().to(device)
model_bbb.load_state_dict(
    torch.load(MODEL_WEIGHTS / "Model_3.pth", map_location=device, weights_only=True)
)
model_bbb.eval()

model_ecg = Model_4().to(device)
model_ecg.load_state_dict(
    torch.load(MODEL_WEIGHTS / "Model_4.pth", map_location=device, weights_only=True)
)
model_ecg.eval()

model = Model_1_3_4().to(device)
model.load_state_dict(
    torch.load(MODEL_WEIGHTS / "Model_1_3_4.pth", map_location=device, weights_only=True)
)

In [ ]:
"""Test model"""
testset = Model_1_3_4_dataset(
    X_test_clinical, Y_test_clinical, X_test_bbb, Y_test_bbb, X_test_ecg, Y_test_ecg
)
testloader = DataLoader(
    testset, batch_size=len(testset), shuffle=False, collate_fn=collate_model_1_3_4
)
auroc, auprc, f1, acc, y_preds, y_batches = test_model_1_3_4(
    model,
    model_1,
    model_bbb,
    model_ecg,
    "Model 1-3-4",
    testloader,
    plot_cm=True,
    plot_PRC=True,
    plot_predictions=True,
    plot_ROC=True,
    return_preds=True,
)
print(
    f"AUROC: {auroc:.3f}, AUCPR: {auprc:.3f}, F1: {f1:.3f}, ACC: {acc:.3f}, Threshold {model.threshold:.3f}"
)

In [ ]:
"""Append to pred_df"""
c_clinical = pd.read_excel(CLIN_CONTROL_DIR)
o_clinical = pd.read_excel(CLIN_OTS_DIR)
c_clinical.sort_values(by="testID", ascending=True, inplace=True)
o_clinical.sort_values(by="testID", ascending=True, inplace=True)

clinical_data_id_validation = np.vstack((c_clinical, o_clinical))[testset_idx]
pred_df_model_1_3_4 = (
    pd.DataFrame(clinical_data_id_validation).drop(columns=[1, 2, 3, 4]).rename(columns={0: "ID"})
)

y_preds_numpy = pd.Series(y_preds.detach().cpu().numpy())
pred_df_model_1_3_4 = pd.concat([pred_df_model_1_3_4, y_preds_numpy], axis=1).rename(
    columns={0: "Model-1-3-4-raw"}
)

y_preds_bin = [1 if val >= model.threshold else 0 for val in y_preds_numpy]
pred_df_model_1_3_4 = pd.concat(
    [pred_df_model_1_3_4, pd.Series(y_preds_bin).rename("Model-1-3-4-pred")], axis=1
)

pred_df_model_1_3_4

In [ ]:
pred_df_combined = pred_df.merge(pred_df_model_1_3_4, on="ID", how="left")
pred_df_combined

In [ ]:
np.testing.assert_array_equal(pred_df_combined["Model-1-3-4-pred"], y_preds_bin)
np.testing.assert_array_equal(pred_df_combined["Ground truth"], y_batches.detach().cpu().numpy())

In [ ]:
pred_df_combined.to_excel(output_path("preds-df.xlsx"))

# Statistics

In [ ]:
import json
import math
from statistics import NormalDist

import numpy as np
import pandas as pd
from MLstatkit import Delong_test
from scipy.stats import binomtest, norm
from sklearn.metrics import confusion_matrix, roc_auc_score
from statsmodels.stats.contingency_tables import mcnemar

In [ ]:
preds_df = pd.read_excel(require_file(output_path("preds-df.xlsx")))

# Reject missing/nonbinary labels rather than silently dropping subjects.
prediction_columns = ["Ground truth", "Model-1-2-pred", "Model-1-3-4-pred"]
if not preds_df[prediction_columns].isin([0, 1]).all().all():
    raise ValueError("Ground truth and model predictions must contain only 0 and 1.")

# Explicit labels preserve TN, FP, FN, TP order even if one class is absent.
counts_1_2 = confusion_matrix(
    preds_df["Ground truth"], preds_df["Model-1-2-pred"], labels=[0, 1]
).ravel()
counts_1_3_4 = confusion_matrix(
    preds_df["Ground truth"], preds_df["Model-1-3-4-pred"], labels=[0, 1]
).ravel()
preds_df

In [ ]:
stats_df = pd.DataFrame(
    columns=[
        "Metric",
        "Statistical test",
        "p-value",
        "CI Method",
        "Difference",
        "Ratio",
        "CI_low",
        "CI_high",
    ]
)

In [ ]:
def paired_correctness_table(y_true, pred_a, pred_b, label):
    """Count paired correct/incorrect predictions within one true class.

    Rows represent model A and columns model B, with correct before incorrect.
    Off-diagonal counts are A correct/B wrong and A wrong/B correct.
    """
    y_true = np.asarray(y_true)
    selected = y_true == label
    correct_a = np.asarray(pred_a)[selected] == label
    correct_b = np.asarray(pred_b)[selected] == label
    return np.array(
        [
            [np.sum(correct_a & correct_b), np.sum(correct_a & ~correct_b)],
            [np.sum(~correct_a & correct_b), np.sum(~correct_a & ~correct_b)],
        ],
        dtype=int,
    )


def tango_ci(b, c, n, confidence=0.95):
    """
    Tango score CI for paired proportions.

    Parameters
    ----------
    b : int
        Number where model A succeeds and model B fails.
    c : int
        Number where model A fails and model B succeeds.
    n : int
        Total number of paired observations.
    confidence : float
        Confidence level, e.g. 0.95.

    Returns
    -------
    (lower, upper)
        CI for p_A - p_B.
    """

    z = norm.ppf(1 - (1 - confidence) / 2)

    # The PropCIs algorithm below works internally with (c-b)/n,
    # i.e. B - A. We negate/reverse at the end to return A - B.

    pa = 2 * n

    # Upper limit for B - A
    if c == n:
        upper = 1.0
    else:
        root = (c - b) / n
        step = 1 - root

        for _ in range(50):
            step /= 2
            x = root + step

            pb = -b - c + (2 * n - c + b) * x
            pc = -b * x * (1 - x)

            q21 = (math.sqrt(pb**2 - 4 * pa * pc) - pb) / (2 * pa)

            score = (c - b - n * x) / math.sqrt(n * (2 * q21 + x * (1 - x)))

            if abs(score) < z:
                root = x

            if step < 1e-7 or abs(z - score) < 1e-6:
                upper = x
                break

    # Lower limit for B - A
    if b == n:
        lower = -1.0
    else:
        root = (c - b) / n
        step = 1 + root

        for _ in range(50):
            step /= 2
            x = root - step

            pb = -b - c + (2 * n - c + b) * x
            pc = -b * x * (1 - x)

            q21 = (math.sqrt(pb**2 - 4 * pa * pc) - pb) / (2 * pa)

            score = (c - b - n * x) / math.sqrt(n * (2 * q21 + x * (1 - x)))

            if abs(score) < z:
                root = x

            if step < 1e-7 or abs(z - score) < 1e-6:
                lower = x
                break

    # Convert B-A interval into A-B interval
    return -upper, -lower

In [ ]:
def CI_metrics(TN, FP, FN, TP):
    """Compute the study confidence intervals from binary confusion-matrix counts.

    TN, FP, FN, and TP are counts for control=0 and OTS=1. Formulas and original
    zero-count behavior are retained; see the research limitations in README.md."""
    sens = binomtest(k=TP, n=TP + FN)
    sens = sens.proportion_ci()
    print(f"Sensitivity {sens.low:.3f}-{sens.high:.3f}")

    spec = binomtest(k=TN, n=TN + FP)
    spec = spec.proportion_ci()
    print(f"Specificity {spec.low:.3f}-{spec.high:.3f}")

In [ ]:
def metrics(y_true, y_pred):
    """Compute the notebook's diagnostic measures from binary labels.

    Inputs use control=0 and OTS=1. Returns the original measure dictionary;
    missing-class and zero-denominator behavior are unchanged."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))

    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    # May become inf/undefined if specificity = 1
    lr_pos = sensitivity / (1 - specificity) if specificity < 1 else np.inf

    lr_neg = (1 - sensitivity) / specificity if specificity > 0 else np.inf

    return {
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "LR_plus": lr_pos,
        "LR_minus": lr_neg,
    }

In [ ]:
def paired_bootstrap(y_true, pred_a, pred_b, metric, n_boot=10000, random_state=42):
    """Resample paired subject predictions to estimate the study metric intervals.

    Preserves pairing across compared models and the original resampling and
    interval formulas. Inputs must represent the same subjects in matching order."""
    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)

    rng = np.random.default_rng(random_state)

    # Positive and negative subject indices
    pos_idx = np.where(y_true == 1)[0]
    neg_idx = np.where(y_true == 0)[0]

    differences = []

    for _ in range(n_boot):
        # Resample subjects WITH replacement within each class
        boot_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
        boot_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)

        idx = np.concatenate([boot_pos, boot_neg])

        # Crucial: use exactly the SAME sampled subjects for A and B
        y_boot = y_true[idx]
        a_boot = pred_a[idx]
        b_boot = pred_b[idx]

        metric_a = metrics(y_boot, a_boot)[metric]
        metric_b = metrics(y_boot, b_boot)[metric]

        diff = metric_a - metric_b

        # Ignore infinite/undefined bootstrap replicates
        if np.isfinite(diff):
            differences.append(diff)

    differences = np.asarray(differences)

    observed_a = metrics(y_true, pred_a)[metric]
    observed_b = metrics(y_true, pred_b)[metric]
    observed_diff = observed_a - observed_b

    # Percentile bootstrap CI
    lower, upper = np.percentile(differences, [2.5, 97.5])

    # Approximate two-sided bootstrap p-value for H0: difference = 0
    p_lower = np.mean(differences <= 0)
    p_upper = np.mean(differences >= 0)
    p_value = min(1, 2 * min(p_lower, p_upper))

    return {
        f"{metric}_A": observed_a,
        f"{metric}_B": observed_b,
        "difference_A_minus_B": observed_diff,
        "ci_95": (lower, upper),
        "bootstrap_p": p_value,
        "n_valid_bootstrap": len(differences),
    }

In [ ]:
def paired_bootstrap_lr_ratio(y_true, pred_a, pred_b, metric, n_boot=10000, random_state=42):
    """
    Paired stratified bootstrap for the ratio:

        LR_A / LR_B

    metric must be:
        "LR_plus" or "LR_minus"

    The bootstrap is performed on the log-ratio scale,
    and the CI is transformed back using exp().
    """

    if metric not in ("LR_plus", "LR_minus"):
        raise ValueError("metric must be 'LR_plus' or 'LR_minus'")

    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)

    rng = np.random.default_rng(random_state)

    # Separate positive and negative subjects
    pos_idx = np.where(y_true == 1)[0]
    neg_idx = np.where(y_true == 0)[0]

    log_ratios = []

    for _ in range(n_boot):
        # Stratified bootstrap:
        # resample positives and negatives separately
        boot_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)

        boot_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)

        idx = np.concatenate([boot_pos, boot_neg])

        # Same subjects used for both models -> paired bootstrap
        y_boot = y_true[idx]
        a_boot = pred_a[idx]
        b_boot = pred_b[idx]

        lr_a = metrics(y_boot, a_boot)[metric]
        lr_b = metrics(y_boot, b_boot)[metric]

        # Log ratio only exists for finite, positive LRs
        if np.isfinite(lr_a) and np.isfinite(lr_b) and lr_a > 0 and lr_b > 0:
            log_ratio = np.log(lr_a / lr_b)
            log_ratios.append(log_ratio)

    log_ratios = np.asarray(log_ratios)

    if len(log_ratios) == 0:
        raise RuntimeError(
            "No valid bootstrap samples. LRs were zero or infinite in all bootstrap replicates."
        )

    # Original observed values
    lr_a_obs = metrics(y_true, pred_a)[metric]
    lr_b_obs = metrics(y_true, pred_b)[metric]

    observed_ratio = lr_a_obs / lr_b_obs

    # Percentile CI on log scale
    lower_log, upper_log = np.percentile(log_ratios, [2.5, 97.5])

    # Transform back to ratio scale
    ci_lower = np.exp(lower_log)
    ci_upper = np.exp(upper_log)

    return {
        "metric": metric,
        "LR_A": lr_a_obs,
        "LR_B": lr_b_obs,
        "ratio_A_over_B": observed_ratio,
        "ci_95": (ci_lower, ci_upper),
        "n_bootstrap": n_boot,
        "n_valid_bootstrap": len(log_ratios),
        "n_invalid_bootstrap": n_boot - len(log_ratios),
    }

In [ ]:
def paired_permutation_lr(y_true, pred_a, pred_b, metric, n_perm=10000, random_state=42):
    """Run the study paired permutation comparison of likelihood ratios.

    Inputs must use matching subject order across models. The original
    randomization, likelihood-ratio formulas, and return values are retained."""
    if metric not in ("LR_plus", "LR_minus"):
        raise ValueError("metric must be 'LR_plus' or 'LR_minus'")

    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)

    rng = np.random.default_rng(random_state)

    # Observed statistic
    lr_a_obs = metrics(y_true, pred_a)[metric]
    lr_b_obs = metrics(y_true, pred_b)[metric]

    if not np.isfinite(lr_a_obs) or not np.isfinite(lr_b_obs) or lr_a_obs <= 0 or lr_b_obs <= 0:
        raise ValueError("Observed LR values must be finite and > 0")

    observed_log_ratio = np.log(lr_a_obs / lr_b_obs)
    observed_stat = abs(observed_log_ratio)

    perm_stats = []

    for _ in range(n_perm):
        # Randomly decide for each subject whether to swap A and B
        swap = rng.random(len(y_true)) < 0.5

        perm_a = pred_a.copy()
        perm_b = pred_b.copy()

        perm_a[swap] = pred_b[swap]
        perm_b[swap] = pred_a[swap]

        lr_a_perm = metrics(y_true, perm_a)[metric]
        lr_b_perm = metrics(y_true, perm_b)[metric]

        # Skip undefined/infinite LR values
        if np.isfinite(lr_a_perm) and np.isfinite(lr_b_perm) and lr_a_perm > 0 and lr_b_perm > 0:
            perm_log_ratio = np.log(lr_a_perm / lr_b_perm)
            perm_stats.append(abs(perm_log_ratio))

    perm_stats = np.asarray(perm_stats)

    # Two-sided permutation p-value
    p_value = (1 + np.sum(perm_stats >= observed_stat)) / (1 + len(perm_stats))

    return {
        "metric": metric,
        "LR_A": lr_a_obs,
        "LR_B": lr_b_obs,
        "ratio_A_over_B": lr_a_obs / lr_b_obs,
        "log_ratio": observed_log_ratio,
        "p_value": p_value,
        "n_valid_permutations": len(perm_stats),
    }

In [ ]:
def likelihood_ratios_ci(tn, fp, fn, tp, confidence=0.95):
    """Estimate likelihood ratios and intervals from tn, fp, fn, tp counts.

    confidence sets the confidence level. Uses the study formulas without
    changing the original handling of zero cells."""
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    lr_plus = sensitivity / (1 - specificity)
    lr_minus = (1 - sensitivity) / specificity

    se_log_lr_plus = math.sqrt(1 / tp - 1 / (tp + fn) + 1 / fp - 1 / (fp + tn))

    se_log_lr_minus = math.sqrt(1 / fn - 1 / (tp + fn) + 1 / tn - 1 / (fp + tn))

    alpha = 1 - confidence
    z = NormalDist().inv_cdf(1 - alpha / 2)

    lr_plus_ci = (
        math.exp(math.log(lr_plus) - z * se_log_lr_plus),
        math.exp(math.log(lr_plus) + z * se_log_lr_plus),
    )

    lr_minus_ci = (
        math.exp(math.log(lr_minus) - z * se_log_lr_minus),
        math.exp(math.log(lr_minus) + z * se_log_lr_minus),
    )

    #     "LR+": lr_plus,
    #     "LR+ CI": lr_plus_ci,
    #     "LR-": lr_minus,
    #     "LR- CI": lr_minus_ci,
    # }
    return [lr_plus, lr_plus_ci, lr_minus, lr_minus_ci]

In [ ]:
def auroc_ci(y_true, y_score, confidence=0.95, n_bootstraps=10000, random_state=42):
    """
    Compute AUROC and bootstrap confidence interval.

    Parameters
    ----------
    y_true : array-like
        Binary true labels (0/1).
    y_score : array-like
        Predicted probabilities or continuous decision scores.
    confidence : float
        Confidence level, default 0.95.
    n_bootstraps : int
        Number of bootstrap samples.
    random_state : int
        Random seed for reproducibility.

    Returns
    -------
    auc : float
        AUROC.
    lower : float
        Lower confidence limit.
    upper : float
        Upper confidence limit.
    """

    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    auc = roc_auc_score(y_true, y_score)

    rng = np.random.default_rng(random_state)
    bootstrapped_aucs = []

    n = len(y_true)

    for _ in range(n_bootstraps):
        indices = rng.integers(0, n, n)

        y_true_boot = y_true[indices]
        y_score_boot = y_score[indices]

        # AUC cannot be computed if bootstrap sample
        # contains only one class
        if len(np.unique(y_true_boot)) < 2:
            continue

        bootstrapped_aucs.append(roc_auc_score(y_true_boot, y_score_boot))

    alpha = 1 - confidence

    lower = np.percentile(bootstrapped_aucs, 100 * alpha / 2)

    upper = np.percentile(bootstrapped_aucs, 100 * (1 - alpha / 2))

    return auc, lower, upper

In [ ]:
"""Sensitivity"""
sensitivity = paired_correctness_table(
    preds_df["Ground truth"],
    preds_df["Model-1-2-pred"],
    preds_df["Model-1-3-4-pred"],
    label=1,
)
results = mcnemar(sensitivity, exact=True, correction=True)

b = sensitivity[0, 1]  # A correct, B wrong
c = sensitivity[1, 0]  # A wrong, B correct
n = sensitivity.sum()

diff = (b - c) / n
ci = tango_ci(b, c, n)

print(
    f"McNemar\n\
    p-value: {results.pvalue:.3f}\n\
    CI: {diff:.3f} ({ci[0]:.3f} - {ci[1]:.3f})"
)

stats_df.loc[0, "Metric"] = "Sensitivity"
stats_df.loc[0, "Statistical test"] = "McNemar"
stats_df.loc[0, "p-value"] = results.pvalue
stats_df.loc[0, "CI Method"] = "Tango's score paired CI"
stats_df.loc[0, "Difference"] = diff
stats_df.loc[0, "CI_low"] = ci[0]
stats_df.loc[0, "CI_high"] = ci[1]

print(
    f"Sensitivity Model-1-2: {metrics(preds_df['Ground truth'], preds_df['Model-1-2-pred'])['Sensitivity']}"
)
print(
    f"Sensitivity Model-1-3-4: {metrics(preds_df['Ground truth'], preds_df['Model-1-3-4-pred'])['Sensitivity']}"
)

In [ ]:
print("\nCI Model-1-2:")
CI_metrics(*counts_1_2)

print("\nCI Model-1-3-4:")
CI_metrics(*counts_1_3_4)

In [ ]:
"""Specificity"""
specificity = paired_correctness_table(
    preds_df["Ground truth"],
    preds_df["Model-1-2-pred"],
    preds_df["Model-1-3-4-pred"],
    label=0,
)

results = mcnemar(specificity, exact=True, correction=True)

b = specificity[0, 1]  # A correct, B wrong
c = specificity[1, 0]  # A wrong, B correct
n = specificity.sum()

diff = (b - c) / n
ci = tango_ci(b, c, n)

print(
    f"McNemar\n\
    p-value: {results.pvalue:.3f}\n\
    CI: {diff:.3f} ({ci[0]:.3f} - {ci[1]:.3f})"
)

stats_df.loc[1, "Metric"] = "Specificity"
stats_df.loc[1, "Statistical test"] = "McNemar"
stats_df.loc[1, "p-value"] = results.pvalue
stats_df.loc[1, "CI Method"] = "Tango's score paired CI"
stats_df.loc[1, "Difference"] = diff
stats_df.loc[1, "CI_low"] = ci[0]
stats_df.loc[1, "CI_high"] = ci[1]

print(
    f"Specificity Model-1-2: {metrics(preds_df['Ground truth'], preds_df['Model-1-2-pred'])['Specificity']}"
)
print(
    f"Specificity Model-1-3-4: {metrics(preds_df['Ground truth'], preds_df['Model-1-3-4-pred'])['Specificity']}"
)

In [ ]:
"""LR+"""
result_ratio = paired_bootstrap_lr_ratio(
    y_true=preds_df["Ground truth"],
    pred_a=preds_df["Model-1-2-pred"],
    pred_b=preds_df["Model-1-3-4-pred"],
    metric="LR_plus",
    n_boot=10000,
    random_state=42,
)

print(json.dumps(result_ratio, indent=2))


result_perm = paired_permutation_lr(
    y_true=preds_df["Ground truth"],
    pred_a=preds_df["Model-1-2-pred"],
    pred_b=preds_df["Model-1-3-4-pred"],
    metric="LR_plus",
)

print(json.dumps(result_perm, indent=2))

stats_df.loc[2, "Metric"] = "LR+"
stats_df.loc[2, "Statistical test"] = "Paired permutation test"
stats_df.loc[2, "p-value"] = result_perm["p_value"]
stats_df.loc[2, "CI Method"] = "Paired bootstrap"
stats_df.loc[2, "Ratio"] = result_ratio["ratio_A_over_B"]
stats_df.loc[2, "CI_low"] = result_ratio["ci_95"][0]
stats_df.loc[2, "CI_high"] = result_ratio["ci_95"][1]

In [ ]:
"""LR-"""
result_ratio = paired_bootstrap_lr_ratio(
    y_true=preds_df["Ground truth"],
    pred_a=preds_df["Model-1-2-pred"],
    pred_b=preds_df["Model-1-3-4-pred"],
    metric="LR_minus",
    n_boot=10000,
    random_state=42,
)

print(json.dumps(result_ratio, indent=2))


result_perm = paired_permutation_lr(
    y_true=preds_df["Ground truth"],
    pred_a=preds_df["Model-1-2-pred"],
    pred_b=preds_df["Model-1-3-4-pred"],
    metric="LR_minus",
)

print(json.dumps(result_perm, indent=2))

stats_df.loc[3, "Metric"] = "LR-"
stats_df.loc[3, "Statistical test"] = "Paired permutation test"
stats_df.loc[3, "p-value"] = result_perm["p_value"]
stats_df.loc[3, "CI Method"] = "Paired bootstrap"
stats_df.loc[3, "Ratio"] = result_ratio["ratio_A_over_B"]
stats_df.loc[3, "CI_low"] = result_ratio["ci_95"][0]
stats_df.loc[3, "CI_high"] = result_ratio["ci_95"][1]

In [ ]:
"""CI for LR+ and LR-"""
lr_plus, lr_plus_ci, lr_minus, lr_minus_ci = likelihood_ratios_ci(*counts_1_2)
print(
    f"Model-1-2:\n\
LR+: {lr_plus} {lr_plus_ci}\n\
LR-: {lr_minus} {lr_minus_ci}"
)
lr_plus, lr_plus_ci, lr_minus, lr_minus_ci = likelihood_ratios_ci(*counts_1_3_4)
print(
    f"Model-1-3-4:\n\
LR+: {lr_plus} {lr_plus_ci}\n\
LR-: {lr_minus} {lr_minus_ci}"
)

In [ ]:
"""AUROC"""
z, p, ci_A, ci_B, auc_A, auc_B, info = Delong_test(
    preds_df["Ground truth"],
    preds_df["Model-1-2-raw"],
    preds_df["Model-1-3-4-raw"],
    alpha=0.95,
    return_ci=True,
    return_auc=True,
    verbose=0,
)

# AUC difference
auc_diff = auc_A - auc_B

# 95% CI for AUC difference
if info["method"] == "delong":
    se_diff = np.sqrt(info["var_diff"])
    zcrit = norm.ppf(0.975)

    ci_diff = (auc_diff - zcrit * se_diff, auc_diff + zcrit * se_diff)

    print(f"AUC A: {auc_A:.3f} (95% CI {ci_A[0]:.3f}-{ci_A[1]:.3f})")
    print(f"AUC B: {auc_B:.3f} (95% CI {ci_B[0]:.3f}-{ci_B[1]:.3f})")
    print(f"AUC difference A-B: {auc_diff:.3f}")
    print(f"95% CI for difference: {ci_diff[0]:.3f} to {ci_diff[1]:.3f}")
    print(f"DeLong z: {z:.3f}")
    print(f"DeLong p-value: {p:.4f}")
else:
    print("MLstatkit used its bootstrap fallback rather than standard DeLong.")

In [ ]:
stats_df.loc[4, "Metric"] = "AUROC"
stats_df.loc[4, "Statistical test"] = "Paired DeLong test"
stats_df.loc[4, "p-value"] = p
stats_df.loc[4, "CI Method"] = "DeLong normal-approximation CI"
stats_df.loc[4, "Difference"] = auc_diff
stats_df.loc[4, "CI_low"] = ci_diff[0]
stats_df.loc[4, "CI_high"] = ci_diff[1]

In [ ]:
"""CI for AUROC"""
auc, lower, upper = auroc_ci(preds_df["Ground truth"], preds_df["Model-1-2-raw"])

print(f"Model-1-2:\nAUROC = {auc:.3f} (95% CI {lower:.3f}–{upper:.3f})")

auc, lower, upper = auroc_ci(preds_df["Ground truth"], preds_df["Model-1-3-4-raw"])

print(f"Model-1-3-4:\nAUROC = {auc:.3f} (95% CI {lower:.3f}–{upper:.3f})")

In [ ]:
stats_df.to_excel(output_path("stats.xlsx"), index=False)